In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


In [2]:
# from pathlib import Path
# import numpy as np
# import pandas as pd
# import matplotlib.pyplot as plt

# # =========================================================
# # 설정: Erangel 폴더(맵별 로그 분리된 CSV) 경로
# # =========================================================
# BASE = Path(r"C:\msys64\home\for\10th\00_Project\04_final\02_parquet_file\03_mapwise_logs_kakao+steam_20260211_20260219")
# MAP_DIR = BASE / "Erangel"

# landing_path  = MAP_DIR / "LogParachuteLanding.csv"
# pos_path      = MAP_DIR / "LogPlayerPosition.csv"
# gs_path       = MAP_DIR / "LogGameStatePeriodic.csv"

# POS_SAMPLE_FRAC = 1.0  # 0.2 등으로 낮추면 빠른 EDA 가능

# # =========================================================
# # 1) 필요한 최소 컬럼만 로드
# # =========================================================
# landing_cols = [
#     "matchId", "_D", "character_accountId",
#     "character_location_x", "character_location_y", "character_location_z",
# ]
# pos_cols = [
#     "matchId", "_D", "elapsedTime", "character_accountId", "character_teamId",
#     "character_location_x", "character_location_y", "character_location_z",
#     "character_isInVehicle", "character_isInBlueZone",
# ]
# gs_cols = [
#     "matchId", "_D", "gameState_elapsedTime",
#     "gameState_safetyZonePosition_x", "gameState_safetyZonePosition_y", "gameState_safetyZoneRadius",
# ]

# landing = pd.read_csv(landing_path, usecols=lambda c: c in landing_cols)
# pos     = pd.read_csv(pos_path,     usecols=lambda c: c in pos_cols)
# gs      = pd.read_csv(gs_path,      usecols=lambda c: c in gs_cols)

# if POS_SAMPLE_FRAC < 1.0:
#     pos = pos.sample(frac=POS_SAMPLE_FRAC, random_state=42)

# # =========================================================
# # 2) 타입/결측 정리 (merge_asof 안전화)
# # =========================================================
# landing["_D"] = pd.to_datetime(landing["_D"], errors="coerce", utc=True)
# pos["_D"]     = pd.to_datetime(pos["_D"],     errors="coerce", utc=True)
# gs["_D"]      = pd.to_datetime(gs["_D"],      errors="coerce", utc=True)

# landing = landing.dropna(subset=["matchId","character_accountId","_D"]).copy()
# pos     = pos.dropna(subset=["matchId","character_accountId","_D"]).copy()
# gs      = gs.dropna(subset=["matchId","_D"]).copy()

# # matchId 혼합 타입 방지
# landing["matchId"] = landing["matchId"].astype(str)
# pos["matchId"]     = pos["matchId"].astype(str)
# gs["matchId"]      = gs["matchId"].astype(str)

# # boolean 정리(문자/0/1 섞여도 mean 계산 가능하게)
# def to_bool(s: pd.Series) -> pd.Series:
#     if str(s.dtype) in ("bool", "boolean"):
#         return s.astype("boolean")
#     return (
#         s.astype(str).str.lower()
#          .map({"true": True, "false": False, "1": True, "0": False, "nan": pd.NA, "none": pd.NA})
#          .astype("boolean")
#     )

# pos["character_isInVehicle"] = to_bool(pos["character_isInVehicle"])
# pos["character_isInBlueZone"] = to_bool(pos["character_isInBlueZone"])

# # =========================================================
# # 3) (A) 매치×유저 피처 생성
# # =========================================================

# # 3-1) Landing → drop table (matchId, accountId 1행)
# landing_sorted = landing.sort_values(["matchId","character_accountId","_D"], kind="mergesort").reset_index(drop=True)
# drop = (
#     landing_sorted
#     .groupby(["matchId","character_accountId"], as_index=False)
#     .first()
#     .rename(columns={
#         "character_location_x": "drop_x",
#         "character_location_y": "drop_y",
#         "character_location_z": "drop_z",
#         "_D": "drop_time",
#     })
# )

# # 3-2) Position ⨝ GameState (merge_asof)
# pos_sorted = pos.sort_values(["matchId","_D"], kind="mergesort").reset_index(drop=True)
# gs_sorted  = gs.sort_values(["matchId","_D"], kind="mergesort").reset_index(drop=True)

# try:
#     # 정상 경로(빠름): by="matchId"
#     pos_gs = pd.merge_asof(
#         pos_sorted,
#         gs_sorted,
#         on="_D",
#         by="matchId",
#         direction="backward",
#         allow_exact_matches=True
#     )
# except ValueError:
#     # fallback(확실): matchId별로 쪼개서 merge_asof
#     print("merge_asof(by=matchId) failed -> fallback to per-match merge_asof")
#     parts = []
#     gs_groups = {mid: g.drop(columns=["matchId"]) for mid, g in gs_sorted.groupby("matchId", sort=False)}
#     # ▲ 핵심 수정: 오른쪽(gs)에서 matchId를 제거해서 matchId_x/y가 생기지 않게 함

#     for mid, gpos in pos_sorted.groupby("matchId", sort=False):
#         ggs = gs_groups.get(mid)
#         if ggs is None or ggs.empty:
#             continue
#         merged = pd.merge_asof(
#             gpos, ggs,
#             on="_D",
#             direction="backward",
#             allow_exact_matches=True
#         )
#         # merged에는 gpos의 matchId가 그대로 남아있음
#         parts.append(merged)

#     pos_gs = pd.concat(parts, ignore_index=True)

# # 혹시라도(다른 이유로) matchId가 suffix로 변했을 때 복구
# if "matchId" not in pos_gs.columns:
#     if "matchId_x" in pos_gs.columns:
#         pos_gs["matchId"] = pos_gs["matchId_x"]
#     elif "matchId_y" in pos_gs.columns:
#         pos_gs["matchId"] = pos_gs["matchId_y"]
#     else:
#         raise RuntimeError("merge 결과에 matchId가 없습니다. 컬럼명을 확인하세요.")

# # 서클 정보 없는 행 제거
# pos_gs = pos_gs.dropna(subset=[
#     "gameState_safetyZonePosition_x",
#     "gameState_safetyZonePosition_y",
#     "gameState_safetyZoneRadius"
# ]).copy()

# # 3-3) safezone 정규화 거리(dist_norm)
# dx = pos_gs["character_location_x"] - pos_gs["gameState_safetyZonePosition_x"]
# dy = pos_gs["character_location_y"] - pos_gs["gameState_safetyZonePosition_y"]
# dist = np.sqrt(dx*dx + dy*dy)

# radius = pos_gs["gameState_safetyZoneRadius"].replace(0, np.nan)
# pos_gs["safe_dist_norm"] = dist / radius

# # 3-4) 매치×유저 집계
# tmp = pos_gs.copy()
# tmp["is_edge"] = tmp["safe_dist_norm"] > 0.8
# tmp["is_center"] = tmp["safe_dist_norm"] < 0.3

# match_user = (
#     tmp.groupby(["matchId","character_accountId"], as_index=False)
#        .agg(
#            pos_samples=("safe_dist_norm","size"),
#            safe_norm_mean=("safe_dist_norm","mean"),
#            safe_norm_std=("safe_dist_norm","std"),
#            edge_ratio=("is_edge","mean"),
#            center_ratio=("is_center","mean"),
#            z_std=("character_location_z","std"),
#            vehicle_ratio=("character_isInVehicle","mean"),
#            bluezone_ratio=("character_isInBlueZone","mean"),
#            elapsed_min=("elapsedTime","min"),
#            elapsed_max=("elapsedTime","max"),
#        )
# )

# # drop 피처 merge(landing 없는 유저는 NaN)
# match_user = match_user.merge(
#     drop[["matchId","character_accountId","drop_x","drop_y","drop_z","drop_time"]],
#     on=["matchId","character_accountId"],
#     how="left"
# )

# # =========================================================
# # 4) (B) 유저 프로필 생성 (accountId 단위)
# # =========================================================
# user_profile = (
#     match_user.groupby("character_accountId", as_index=False)
#               .agg(
#                   matches=("matchId","nunique"),
#                   safe_norm_mean=("safe_norm_mean","mean"),
#                   safe_norm_std=("safe_norm_mean","std"),
#                   edge_ratio=("edge_ratio","mean"),
#                   center_ratio=("center_ratio","mean"),
#                   vehicle_ratio=("vehicle_ratio","mean"),
#                   bluezone_ratio=("bluezone_ratio","mean"),
#                   z_std=("z_std","mean"),
#                   drop_x_mean=("drop_x","mean"),
#                   drop_y_mean=("drop_y","mean"),
#                   drop_x_std=("drop_x","std"),
#                   drop_y_std=("drop_y","std"),
#               )
# )

# # =========================================================
# # 5) EDA 시각화
# # =========================================================
# plt.figure()
# plt.scatter(match_user["safe_norm_mean"], match_user["vehicle_ratio"], s=5, alpha = 0.4)
# plt.xlabel("safe_dist_norm mean (match-user)")
# plt.ylabel("vehicle_ratio (match-user)")
# plt.title("Positioning vs Vehicle Dependency (Match-User)")
# plt.show()

# plt.figure()
# plt.scatter(user_profile["edge_ratio"], user_profile["center_ratio"], s=8, alpha = 0.4)
# plt.xlabel("edge_ratio (user)")
# plt.ylabel("center_ratio (user)")
# plt.title("Edge vs Center Tendency (User Profile)")
# plt.show()

# plt.figure()
# plt.scatter(user_profile["vehicle_ratio"], user_profile["bluezone_ratio"], s=8, alpha = 0.4)
# plt.xlabel("vehicle_ratio (user)")
# plt.ylabel("bluezone_ratio (user)")
# plt.title("Vehicle Dependency vs Bluezone (User Profile)")
# plt.show()

# plt.figure()
# plt.hist(user_profile["matches"], bins=30)
# plt.xlabel("matches per user")
# plt.ylabel("count of users")
# plt.title("User Activity Distribution")
# plt.show()

# print("match_user shape:", match_user.shape)
# print("user_profile shape:", user_profile.shape)
# print("\nmatch_user head:")
# print(match_user.head(3))
# print("\nuser_profile head:")
# print(user_profile.head(3))

피처         설명

|피처|설명|
|:---:|:---:|
|drop_distance_from_path    |비행기 경로에서 낙하 지점까지 수직 거리
|early_enemy_density        |낙하 직후 반경 500m 내 적 수
|rotation_timing_score      |자기장 선점 vs 후행 비율 (0=선점, 1=후행)
|vehicle_use_ratio	        |차량 이동 비율
|bluezone_exposure_ratio	|블루존 체류 비율
|safezone_proximity_mean	|안전구역 중심까지 평균 거리
|safezone_edge_ratio	    |안전구역 반경 대비 상대 거리
|altitude_variance	        |고도 변화량 (지형 활용도)
|max_vehicle_distance	    |낙하 지점 대비 최대 차량 이동 거리

In [3]:
df_erangel_gamestate = pd.read_csv("C:/Users/qkrtl/10th/00_Project/04_final/02_parquet_file/03_mapwise_logs_kakao+steam_20260211_20260219/Erangel/LogGameStatePeriodic.csv")
df_erangel_gamestate.T

,0,1,2,3,4,5,6,7,8,9,...,367311,367312,367313,367314,367315,367316,367317,367318,367319,367320
matchId,047584e1-2c6e-4ae4-bb66-e263beb868aa,047584e1-2c6e-4ae4-bb66-e263beb868aa,047584e1-2c6e-4ae4-bb66-e263beb868aa,047584e1-2c6e-4ae4-bb66-e263beb868aa,047584e1-2c6e-4ae4-bb66-e263beb868aa,047584e1-2c6e-4ae4-bb66-e263beb868aa,047584e1-2c6e-4ae4-bb66-e263beb868aa,047584e1-2c6e-4ae4-bb66-e263beb868aa,047584e1-2c6e-4ae4-bb66-e263beb868aa,047584e1-2c6e-4ae4-bb66-e263beb868aa,...,fe1ab49f-7765-4ca8-b62f-c6486d7ca86f,fe1ab49f-7765-4ca8-b62f-c6486d7ca86f,fe1ab49f-7765-4ca8-b62f-c6486d7ca86f,fe1ab49f-7765-4ca8-b62f-c6486d7ca86f,fe1ab49f-7765-4ca8-b62f-c6486d7ca86f,fe1ab49f-7765-4ca8-b62f-c6486d7ca86f,fe1ab49f-7765-4ca8-b62f-c6486d7ca86f,fe1ab49f-7765-4ca8-b62f-c6486d7ca86f,fe1ab49f-7765-4ca8-b62f-c6486d7ca86f,fe1ab49f-7765-4ca8-b62f-c6486d7ca86f
_D,2026-02-10T14:47:16.358Z,2026-02-10T14:47:26.192Z,2026-02-10T14:47:36.186Z,2026-02-10T14:47:46.164Z,2026-02-10T14:47:56.172Z,2026-02-10T14:48:06.170Z,2026-02-10T14:48:16.179Z,2026-02-10T14:48:26.213Z,2026-02-10T14:48:36.188Z,2026-02-10T14:48:46.173Z,...,2026-02-18T04:58:58.521Z,2026-02-18T04:59:08.509Z,2026-02-18T04:59:18.496Z,2026-02-18T04:59:28.517Z,2026-02-18T04:59:38.505Z,2026-02-18T04:59:48.493Z,2026-02-18T04:59:58.513Z,2026-02-18T05:00:08.501Z,2026-02-18T05:00:18.521Z,2026-02-18T05:00:28.509Z
_T,LogGameStatePeriodic,LogGameStatePeriodic,LogGameStatePeriodic,LogGameStatePeriodic,LogGameStatePeriodic,LogGameStatePeriodic,LogGameStatePeriodic,LogGameStatePeriodic,LogGameStatePeriodic,LogGameStatePeriodic,...,LogGameStatePeriodic,LogGameStatePeriodic,LogGameStatePeriodic,LogGameStatePeriodic,LogGameStatePeriodic,LogGameStatePeriodic,LogGameStatePeriodic,LogGameStatePeriodic,LogGameStatePeriodic,LogGameStatePeriodic
common_isGame,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,1.0,...,7.0,7.5,7.5,7.5,7.5,7.5,7.5,7.5,7.5,7.5
gameState_elapsedTime,10.0,20.0,29.0,39.0,49.0,59.0,69.0,79.0,89.0,99.0,...,1576.0,1586.0,1596.0,1606.0,1616.0,1626.0,1636.0,1646.0,1656.0,1666.0
gameState_numStartTeams,16.0,16.0,16.0,16.0,16.0,16.0,16.0,16.0,16.0,16.0,...,14.0,14.0,14.0,14.0,14.0,14.0,14.0,14.0,14.0,14.0
gameState_numAliveTeams,16.0,16.0,16.0,16.0,16.0,16.0,16.0,16.0,16.0,16.0,...,3.0,3.0,3.0,3.0,2.0,2.0,2.0,2.0,2.0,1.0
gameState_numParticipatedTeams,16.0,16.0,16.0,16.0,16.0,16.0,16.0,16.0,16.0,16.0,...,3.0,3.0,3.0,3.0,2.0,2.0,2.0,2.0,2.0,1.0
gameState_numJoinPlayers,61.0,61.0,61.0,61.0,61.0,61.0,61.0,61.0,61.0,61.0,...,12.0,12.0,12.0,12.0,12.0,12.0,11.0,11.0,11.0,11.0
gameState_numStartPlayers,61.0,61.0,61.0,61.0,61.0,61.0,61.0,61.0,61.0,61.0,...,56.0,56.0,56.0,56.0,56.0,56.0,56.0,56.0,56.0,56.0


In [4]:
df_erangel_gamestate.dtypes

matchId                                     str
_D                                          str
_T                                          str
common_isGame                           float64
gameState_elapsedTime                   float64
gameState_numStartTeams                 float64
gameState_numAliveTeams                 float64
gameState_numParticipatedTeams          float64
gameState_numJoinPlayers                float64
gameState_numStartPlayers               float64
gameState_numAlivePlayers               float64
gameState_numParticipatedPlayers        float64
gameState_safetyZonePosition_x          float64
gameState_safetyZonePosition_y          float64
gameState_safetyZonePosition_z          float64
gameState_safetyZoneRadius              float64
gameState_poisonGasWarningPosition_x    float64
gameState_poisonGasWarningPosition_y    float64
gameState_poisonGasWarningPosition_z    float64
gameState_poisonGasWarningRadius        float64
gameState_redZonePosition_x             

In [5]:
from pathlib import Path
import numpy as np
import pandas as pd

# =========================================================
# 0) 경로 설정 (맵별 로그 폴더)
# =========================================================
BASE = Path("C:/Users/qkrtl/10th/00_Project/04_final/02_parquet_file/03_mapwise_logs_kakao+steam_20260211_20260219")
MAP_DIR = BASE / "Erangel"   # 필요시 변경: Miramar / Rondo / Sanhok / Taego

# 필수
landing_path = MAP_DIR / "LogParachuteLanding.csv"
pos_path     = MAP_DIR / "LogPlayerPosition.csv"
gs_path      = MAP_DIR / "LogGameStatePeriodic.csv"

# 선택(있으면 사용)
veh_leave_path = MAP_DIR / "LogVehicleLeave.csv"
phase_path     = MAP_DIR / "LogPhaseChange.csv"

# =========================================================
# 1) 로드 (필요 컬럼만)
# =========================================================
landing_cols = [
    "matchId", "_D", "character_accountId",
    "character_location_x", "character_location_y", "character_location_z",
    "distance"
]

pos_cols = [
    "matchId", "_D", "elapsedTime", "character_accountId", "character_teamId",
    "character_location_x", "character_location_y", "character_location_z",
    "character_isInVehicle", "character_isInBlueZone"
]

gs_cols = [
    "matchId", "_D", "gameState_elapsedTime",
    "gameState_safetyZonePosition_x", "gameState_safetyZonePosition_y", "gameState_safetyZoneRadius",
    "gameState_poisonGasWarningPosition_x", "gameState_poisonGasWarningPosition_y", "gameState_poisonGasWarningRadius",
]

veh_leave_cols = [
    "matchId", "_D", "character_accountId",
    "vehicle_location_x", "vehicle_location_y", "vehicle_location_z",
    "rideDistance", "maxSpeed"
]

phase_cols = [
    "matchId", "_D", "phase"
]

landing = pd.read_csv(landing_path, usecols=lambda c: c in landing_cols)
pos     = pd.read_csv(pos_path,     usecols=lambda c: c in pos_cols)
gs      = pd.read_csv(gs_path,      usecols=lambda c: c in gs_cols)

veh_leave = None
if veh_leave_path.exists():
    veh_leave = pd.read_csv(veh_leave_path, usecols=lambda c: c in veh_leave_cols)

phase = None
if phase_path.exists():
    phase = pd.read_csv(phase_path, usecols=lambda c: c in phase_cols)

# =========================================================
# 2) 기본 전처리 (데이터 타입 변환 / 키 정리)
# =========================================================
def prep_common(df, account_col=None):
    df = df.copy()
    if "_D" in df.columns:
        df["_D"] = pd.to_datetime(df["_D"], errors="coerce", utc=True)
    if "matchId" in df.columns:
        df["matchId"] = df["matchId"].astype(str)
    if account_col and account_col in df.columns:
        df = df.rename(columns={account_col: "accountId"})
        df["accountId"] = df["accountId"].astype(str)
    return df

landing = prep_common(landing, account_col="character_accountId")
pos     = prep_common(pos,     account_col="character_accountId")
gs      = prep_common(gs)
if veh_leave is not None:
    veh_leave = prep_common(veh_leave, account_col="character_accountId")
if phase is not None:
    phase = prep_common(phase)

# 결측 제거
landing = landing.dropna(subset=["matchId", "accountId", "_D", "character_location_x", "character_location_y"]).copy()
pos     = pos.dropna(subset=["matchId", "accountId", "_D", "character_location_x", "character_location_y"]).copy()
gs      = gs.dropna(subset=["matchId", "_D", "gameState_safetyZonePosition_x", "gameState_safetyZonePosition_y", "gameState_safetyZoneRadius"]).copy()

if veh_leave is not None:
    veh_leave = veh_leave.dropna(subset=["matchId", "accountId", "_D"]).copy()

if phase is not None:
    phase = phase.dropna(subset=["matchId", "_D"]).copy()

# bool 변환
def to_bool(s: pd.Series) -> pd.Series:
    if str(s.dtype) in ("bool", "boolean"):
        return s.astype("boolean")
    return (
        s.astype(str).str.lower()
         .map({"true": True, "false": False, "1": True, "0": False, "nan": pd.NA, "none": pd.NA})
         .astype("boolean")
    )

if "character_isInVehicle" in pos.columns:
    pos["character_isInVehicle"] = to_bool(pos["character_isInVehicle"])
if "character_isInBlueZone" in pos.columns:
    pos["character_isInBlueZone"] = to_bool(pos["character_isInBlueZone"])

# =========================================================
# 3) 기준 테이블: landing_first (matchId, accountId 1행)
# =========================================================
landing_first = (
    landing.sort_values(["matchId", "accountId", "_D"], kind="mergesort")
           .groupby(["matchId", "accountId"], as_index=False)
           .first()
           .copy()
)

landing_first = landing_first.rename(columns={
    "character_location_x": "drop_x",
    "character_location_y": "drop_y",
    "character_location_z": "drop_z",
    "_D": "drop_time",
    "distance": "parachute_distance"
})

# =========================================================
# 4) Feature 1: drop_distance_from_path
#    (초반 position으로 비행기 경로 직선 추정 -> 낙하지점까지 수직거리)
# =========================================================
# NOTE: elapsedTime 기준은 데이터 보며 조정 (초기 30초 예시)
pos_early = pos[pos["elapsedTime"].notna()].copy()
pos_early = pos_early[pos_early["elapsedTime"] <= 30]

def fit_flight_line_and_drop_distance(pos_early_df, landing_df):
    out = []
    for mid, gpos in pos_early_df.groupby("matchId", sort=False):
        gdrop = landing_df[landing_df["matchId"] == mid]
        if gdrop.empty or len(gpos) < 2:
            continue

        x = gpos["character_location_x"].to_numpy(dtype=float)
        y = gpos["character_location_y"].to_numpy(dtype=float)

        # 축 분산 큰 쪽 기준으로 피팅(수직선 불안정 회피)
        if np.nanstd(x) >= np.nanstd(y):
            a, b = np.polyfit(x, y, 1)  # y = ax + b
            dx = gdrop["drop_x"].to_numpy(dtype=float)
            dy = gdrop["drop_y"].to_numpy(dtype=float)
            dist = np.abs(a * dx - dy + b) / np.sqrt(a*a + 1.0)
        else:
            a, b = np.polyfit(y, x, 1)  # x = ay + b
            dx = gdrop["drop_x"].to_numpy(dtype=float)
            dy = gdrop["drop_y"].to_numpy(dtype=float)
            dist = np.abs(a * dy - dx + b) / np.sqrt(a*a + 1.0)

        tmp = gdrop[["matchId", "accountId"]].copy()
        tmp["drop_distance_from_path"] = dist
        out.append(tmp)

    if not out:
        return pd.DataFrame(columns=["matchId", "accountId", "drop_distance_from_path"])
    return pd.concat(out, ignore_index=True)

feat_drop_path = fit_flight_line_and_drop_distance(pos_early, landing_first)

# =========================================================
# 5) Feature 2: early_enemy_density
#    (낙하 직후 반경 500m 내 적 수)
# =========================================================
def compute_early_enemy_density(landing_df, radius=500.0):
    rows = []
    for mid, g in landing_df.groupby("matchId", sort=False):
        coords = g[["drop_x", "drop_y"]].to_numpy(dtype=float)
        accs = g["accountId"].to_numpy()

        if len(coords) == 0:
            continue

        diff = coords[:, None, :] - coords[None, :, :]
        dist = np.sqrt((diff ** 2).sum(axis=2))
        within = (dist <= radius)
        np.fill_diagonal(within, False)

        tmp = pd.DataFrame({
            "matchId": mid,
            "accountId": accs,
            "early_enemy_density": within.sum(axis=1)
        })
        rows.append(tmp)

    if not rows:
        return pd.DataFrame(columns=["matchId", "accountId", "early_enemy_density"])
    return pd.concat(rows, ignore_index=True)

feat_density = compute_early_enemy_density(landing_first, radius=500.0)

# =========================================================
# 6) Feature 3~5~8: position 기반 (merge 전)
#    - vehicle_use_ratio
#    - bluezone_exposure_ratio
#    - altitude_variance
#    - (추가) altitude_std
# =========================================================
feat_pos_basic = (
    pos.groupby(["matchId", "accountId"], as_index=False)
       .agg(
           vehicle_use_ratio=("character_isInVehicle", "mean"),
           bluezone_exposure_ratio=("character_isInBlueZone", "mean"),
           altitude_variance=("character_location_z", "var"),
           altitude_std=("character_location_z", "std"),
           pos_samples=("character_location_z", "size"),
       )
)

# =========================================================
# 7) Feature 9: max_vehicle_distance
#    (우선순위: veh_leave 사용 > 없으면 position의 차량탑승 좌표로 근사)
# =========================================================
if veh_leave is not None and {"vehicle_location_x", "vehicle_location_y"}.issubset(set(veh_leave.columns)):
    veh_tmp = veh_leave.merge(
        landing_first[["matchId", "accountId", "drop_x", "drop_y"]],
        on=["matchId", "accountId"], how="left"
    )
    dx = veh_tmp["vehicle_location_x"] - veh_tmp["drop_x"]
    dy = veh_tmp["vehicle_location_y"] - veh_tmp["drop_y"]
    veh_tmp["veh_dist_from_drop"] = np.sqrt(dx*dx + dy*dy)

    feat_max_vehicle_dist = (
        veh_tmp.groupby(["matchId", "accountId"], as_index=False)["veh_dist_from_drop"]
               .max()
               .rename(columns={"veh_dist_from_drop": "max_vehicle_distance"})
    )
else:
    # fallback: position에서 차량 탑승 시점 좌표 사용
    pos_vehicle = pos[pos["character_isInVehicle"] == True].copy()
    pos_vehicle = pos_vehicle.merge(
        landing_first[["matchId", "accountId", "drop_x", "drop_y"]],
        on=["matchId", "accountId"], how="left"
    )
    dx = pos_vehicle["character_location_x"] - pos_vehicle["drop_x"]
    dy = pos_vehicle["character_location_y"] - pos_vehicle["drop_y"]
    pos_vehicle["veh_dist_from_drop"] = np.sqrt(dx*dx + dy*dy)

    feat_max_vehicle_dist = (
        pos_vehicle.groupby(["matchId", "accountId"], as_index=False)["veh_dist_from_drop"]
                  .max()
                  .rename(columns={"veh_dist_from_drop": "max_vehicle_distance"})
    )

# =========================================================
# 8) Position ⨝ GameState merge_asof (필수 merge 구간)
#    - rotation_timing_score
#    - safezone_proximity_mean
#    - safezone_edge_ratio
# =========================================================
pos_sorted = pos.sort_values(["matchId", "_D"], kind="mergesort").reset_index(drop=True)
gs_sorted  = gs.sort_values(["matchId", "_D"], kind="mergesort").reset_index(drop=True)

try:
    pos_gs = pd.merge_asof(
        pos_sorted,
        gs_sorted,
        on="_D",
        by="matchId",
        direction="backward",
        allow_exact_matches=True,
        tolerance=pd.Timedelta("15s")   # GameStatePeriodic 주기 고려(조정 가능)
    )
except ValueError:
    # fallback: matchId별 merge_asof
    parts = []
    gs_groups = {mid: g.drop(columns=["matchId"]) for mid, g in gs_sorted.groupby("matchId", sort=False)}
    for mid, gpos in pos_sorted.groupby("matchId", sort=False):
        ggs = gs_groups.get(mid)
        if ggs is None or ggs.empty:
            continue
        merged = pd.merge_asof(
            gpos, ggs,
            on="_D",
            direction="backward",
            allow_exact_matches=True,
            tolerance=pd.Timedelta("15s")
        )
        parts.append(merged)
    pos_gs = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()

if not pos_gs.empty and "matchId" not in pos_gs.columns:
    if "matchId_x" in pos_gs.columns:
        pos_gs["matchId"] = pos_gs["matchId_x"]
    elif "matchId_y" in pos_gs.columns:
        pos_gs["matchId"] = pos_gs["matchId_y"]

# 서클 정보가 붙은 행만 유지
if not pos_gs.empty:
    pos_gs = pos_gs.dropna(subset=[
        "gameState_safetyZonePosition_x",
        "gameState_safetyZonePosition_y",
        "gameState_safetyZoneRadius"
    ]).copy()

# =========================================================
# 9) Feature 6~7: safezone 관련 피처
#    - safezone_proximity_mean
#    - safezone_edge_ratio
# =========================================================
if not pos_gs.empty:
    dx = pos_gs["character_location_x"] - pos_gs["gameState_safetyZonePosition_x"]
    dy = pos_gs["character_location_y"] - pos_gs["gameState_safetyZonePosition_y"]
    pos_gs["safezone_dist"] = np.sqrt(dx*dx + dy*dy)

    radius = pos_gs["gameState_safetyZoneRadius"].replace(0, np.nan)
    pos_gs["safezone_dist_norm"] = pos_gs["safezone_dist"] / radius

    # edge ratio 정의: dist_norm > 0.8 비율 (EDA 보며 조정 가능)
    pos_gs["is_edge"] = pos_gs["safezone_dist_norm"] > 0.8

    feat_safezone = (
        pos_gs.groupby(["matchId", "accountId"], as_index=False)
              .agg(
                  safezone_proximity_mean=("safezone_dist_norm", "mean"),   # 정규화 평균거리
                  safezone_edge_ratio=("is_edge", "mean"),
              )
    )
else:
    feat_safezone = pd.DataFrame(columns=["matchId", "accountId", "safezone_proximity_mean", "safezone_edge_ratio"])

# =========================================================
# 10) Feature 4: rotation_timing_score
#    정의(간이 버전):
#      - 각 match-user에 대해, safezone_dist_norm 시계열의 감소(=서클 쪽으로 접근) 시점을 관찰
#      - 매치 내 상대시간 기준으로 "언제 접근을 시작하는가"를 0~1로 정규화
#      - 0=선점(이른 접근), 1=후행(늦은 접근)
#
#    NOTE: PhaseChange를 함께 쓰면 더 정확해짐(phase별 지연시간 평균)
# =========================================================
def compute_rotation_timing_score(pos_gs_df: pd.DataFrame) -> pd.DataFrame:
    if pos_gs_df.empty:
        return pd.DataFrame(columns=["matchId", "accountId", "rotation_timing_score"])

    rows = []
    grp = pos_gs_df.sort_values(["matchId", "accountId", "_D"]).groupby(["matchId", "accountId"], sort=False)

    for (mid, aid), g in grp:
        g = g.copy()
        # elapsedTime 우선, 없으면 _D 기반 초 차이로 대체
        if "elapsedTime" in g.columns and g["elapsedTime"].notna().sum() > 1:
            t = g["elapsedTime"].to_numpy(dtype=float)
        else:
            t = (g["_D"] - g["_D"].min()).dt.total_seconds().to_numpy(dtype=float)

        d = g["safezone_dist_norm"].to_numpy(dtype=float)

        # 너무 짧으면 계산 불가
        if len(g) < 3 or np.all(np.isnan(d)):
            rows.append([mid, aid, np.nan])
            continue

        # 접근 시작 탐지: dist_norm 감소 추세 시작
        # 간단히 diff < -eps 가 처음 발생한 시점 사용
        dd = np.diff(d)
        valid_idx = np.where(dd < -0.02)[0]  # 임계값은 EDA로 조정
        if len(valid_idx) == 0:
            # 접근 시작 감지 실패 -> 중립값 또는 결측
            score = np.nan
        else:
            # diff index i는 t[i] -> t[i+1] 변화, 시작시점은 t[i+1]로 취급
            start_t = t[valid_idx[0] + 1]
            t_min = np.nanmin(t)
            t_max = np.nanmax(t)
            if np.isfinite(t_min) and np.isfinite(t_max) and t_max > t_min:
                score = (start_t - t_min) / (t_max - t_min)  # 0~1 정규화
            else:
                score = np.nan

        rows.append([mid, aid, score])

    return pd.DataFrame(rows, columns=["matchId", "accountId", "rotation_timing_score"])

feat_rotation = compute_rotation_timing_score(pos_gs)

# =========================================================
# 11) 최종 피처 테이블 결합 (matchId, accountId 기준)
# =========================================================
features_match_user = landing_first[[
    "matchId", "accountId",
    "drop_time", "drop_x", "drop_y", "drop_z",
    "parachute_distance"
]].copy()

for feat_df in [
    feat_drop_path,
    feat_density,
    feat_pos_basic,
    feat_max_vehicle_dist,
    feat_safezone,
    feat_rotation,
]:
    features_match_user = features_match_user.merge(
        feat_df, on=["matchId", "accountId"], how="left"
    )

# =========================================================
# 12) 네가 정의한 피처명만 추려서 보기 (요청한 9개 + 참고 컬럼)
# =========================================================
requested_cols = [
    "matchId", "accountId",
    "drop_distance_from_path",
    "early_enemy_density",
    "rotation_timing_score",
    "vehicle_use_ratio",
    "bluezone_exposure_ratio",
    "safezone_proximity_mean",
    "safezone_edge_ratio",
    "altitude_variance",
    "max_vehicle_distance",
]

# 없는 컬럼 방지
requested_cols = [c for c in requested_cols if c in features_match_user.columns]
features_view = features_match_user[requested_cols].copy()

print("features_match_user shape:", features_match_user.shape)
print("features_view shape:", features_view.shape)
print("\nNull ratio (requested features):")
print(features_view.isna().mean().sort_values(ascending=False))

features_view

features_match_user shape: (148804, 18)
features_view shape: (148804, 11)

Null ratio (requested features):
rotation_timing_score      0.003649
safezone_proximity_mean    0.000517
safezone_edge_ratio        0.000517
drop_distance_from_path    0.000000
accountId                  0.000000
matchId                    0.000000
early_enemy_density        0.000000
bluezone_exposure_ratio    0.000000
vehicle_use_ratio          0.000000
altitude_variance          0.000000
max_vehicle_distance       0.000000
dtype: float64


,matchId,accountId,drop_distance_from_path,early_enemy_density,rotation_timing_score,vehicle_use_ratio,bluezone_exposure_ratio,safezone_proximity_mean,safezone_edge_ratio,altitude_variance,max_vehicle_distance
0,001491cf-ad60-4ebc-b6a1-c86bc05959fd,account.00ae636f3f29443490716d28017686d2,152799.087565,0,0.005562,0.185185,0.116402,0.672490,0.285714,7.079214e+08,164003.650062
1,001491cf-ad60-4ebc-b6a1-c86bc05959fd,account.04550355e52549ea99457094928c0316,23043.979341,0,0.007225,0.138462,0.046154,0.599718,0.300813,7.814969e+08,471343.824630
2,001491cf-ad60-4ebc-b6a1-c86bc05959fd,account.09c7523488a6456abbee4404309fa9b4,23919.139096,0,0.007315,0.451128,0.0,0.410171,0.047619,6.544758e+08,394739.271133
3,001491cf-ad60-4ebc-b6a1-c86bc05959fd,account.11232838286e42cab753a80e7d67dbef,52655.253437,0,0.033557,0.184211,0.0,0.138339,0.000000,2.254545e+09,84767.215571
4,001491cf-ad60-4ebc-b6a1-c86bc05959fd,account.1ddef90833c34149a8aeef6d6b566e26,47827.076519,0,0.016584,0.072464,0.130435,0.561798,0.177419,1.720955e+09,70374.253223
...,...,...,...,...,...,...,...,...,...,...,...
148799,ffd8f1e1-4a98-456d-a9e6-e06251cdbea9,account.ea825a4d327c4f5095718884baf72158,460.514518,0,0.017483,0.044118,0.014706,0.326093,0.050847,1.235701e+09,101877.992595
148800,ffd8f1e1-4a98-456d-a9e6-e06251cdbea9,account.f0a6f8d1e24c4b1881bb826ac3d82fd0,317484.959570,0,0.019120,0.16129,0.0,0.478814,0.000000,2.187561e+09,171796.584975
148801,ffd8f1e1-4a98-456d-a9e6-e06251cdbea9,account.f0d3c4aaa34142eeb9535ba6e8142e35,123339.924504,0,0.006983,0.272727,0.0,0.586898,0.095890,6.600605e+08,361951.350008
148802,ffd8f1e1-4a98-456d-a9e6-e06251cdbea9,account.f54aaf14429943cbab09c30ebc66812b,16072.284061,0,0.017483,0.415385,0.0,0.523011,0.016949,9.951767e+08,271039.849948


In [6]:
features_view.T

,0,1,2,3,4,5,6,7,8,9,...,148794,148795,148796,148797,148798,148799,148800,148801,148802,148803
matchId,001491cf-ad60-4ebc-b6a1-c86bc05959fd,001491cf-ad60-4ebc-b6a1-c86bc05959fd,001491cf-ad60-4ebc-b6a1-c86bc05959fd,001491cf-ad60-4ebc-b6a1-c86bc05959fd,001491cf-ad60-4ebc-b6a1-c86bc05959fd,001491cf-ad60-4ebc-b6a1-c86bc05959fd,001491cf-ad60-4ebc-b6a1-c86bc05959fd,001491cf-ad60-4ebc-b6a1-c86bc05959fd,001491cf-ad60-4ebc-b6a1-c86bc05959fd,001491cf-ad60-4ebc-b6a1-c86bc05959fd,...,ffd8f1e1-4a98-456d-a9e6-e06251cdbea9,ffd8f1e1-4a98-456d-a9e6-e06251cdbea9,ffd8f1e1-4a98-456d-a9e6-e06251cdbea9,ffd8f1e1-4a98-456d-a9e6-e06251cdbea9,ffd8f1e1-4a98-456d-a9e6-e06251cdbea9,ffd8f1e1-4a98-456d-a9e6-e06251cdbea9,ffd8f1e1-4a98-456d-a9e6-e06251cdbea9,ffd8f1e1-4a98-456d-a9e6-e06251cdbea9,ffd8f1e1-4a98-456d-a9e6-e06251cdbea9,ffd8f1e1-4a98-456d-a9e6-e06251cdbea9
accountId,account.00ae636f3f29443490716d28017686d2,account.04550355e52549ea99457094928c0316,account.09c7523488a6456abbee4404309fa9b4,account.11232838286e42cab753a80e7d67dbef,account.1ddef90833c34149a8aeef6d6b566e26,account.23931a30c48041c180cccabc6d663610,account.263c3e56b22c4f3d8066119499641066,account.297f06b4123644f7b6b8e711a78a951c,account.363b850923a74180a4eca2205182d331,account.36cf230a19fa4a0080f876af0107bff7,...,account.df6cd7f0c2564f8ba411c493b1cb1887,account.e0818362272f4d8cba82c51e1e8efcd9,account.e32f77c6cde2461eb42e81361c4b1bf1,account.e9b488acacc041d2a8f181afc1f29812,account.e9c13c0ecb9b4aa2a591289b32e15bde,account.ea825a4d327c4f5095718884baf72158,account.f0a6f8d1e24c4b1881bb826ac3d82fd0,account.f0d3c4aaa34142eeb9535ba6e8142e35,account.f54aaf14429943cbab09c30ebc66812b,account.fdbaf316a9cf4cd58e3d57ab14771ac2
drop_distance_from_path,152799.087565,23043.979341,23919.139096,52655.253437,47827.076519,152931.603036,14330.748597,150327.179067,187040.307284,206176.433305,...,105047.965282,103215.731265,99977.138618,179314.749643,24516.640241,460.514518,317484.95957,123339.924504,16072.284061,14050.474515
early_enemy_density,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
rotation_timing_score,0.005562,0.007225,0.007315,0.033557,0.016584,0.005718,0.019084,0.005441,0.015337,0.503067,...,0.007032,0.009234,0.009671,0.007622,0.008905,0.017483,0.01912,0.006983,0.017483,0.017483
vehicle_use_ratio,0.185185,0.138462,0.451128,0.184211,0.072464,0.248649,0.3,0.248705,0.364865,0.173333,...,0.163399,0.325,0.486957,0.27972,0.196721,0.044118,0.16129,0.272727,0.415385,0.044118
bluezone_exposure_ratio,0.116402,0.046154,0.0,0.0,0.130435,0.102703,0.0,0.222798,0.0,0.0,...,0.0,0.033333,0.0,0.0,0.0,0.014706,0.0,0.0,0.0,0.029412
safezone_proximity_mean,0.67249,0.599718,0.410171,0.138339,0.561798,0.660521,0.541647,0.772248,0.366646,0.41798,...,0.354394,0.373208,0.352952,0.60984,0.463733,0.326093,0.478814,0.586898,0.523011,0.360424
safezone_edge_ratio,0.285714,0.300813,0.047619,0.0,0.177419,0.338983,0.0,0.301075,0.0,0.0,...,0.027586,0.126126,0.09434,0.253731,0.0,0.050847,0.0,0.09589,0.016949,0.084746
altitude_variance,707921391.414622,781496924.684501,654475757.931259,2254545356.766067,1720954822.215166,668270723.672646,2299620042.907699,667649741.394857,801870034.834465,830953412.851743,...,703939136.423324,952529854.345698,983449111.836972,471610985.036669,1074974051.076596,1235701493.460996,2187560624.525437,660060538.421203,995176729.446466,1240583773.410311


In [7]:
# features_match_user = 전처리 완료, features_view = 요약
features_match_user.T

,0,1,2,3,4,5,6,7,8,9,...,148794,148795,148796,148797,148798,148799,148800,148801,148802,148803
matchId,001491cf-ad60-4ebc-b6a1-c86bc05959fd,001491cf-ad60-4ebc-b6a1-c86bc05959fd,001491cf-ad60-4ebc-b6a1-c86bc05959fd,001491cf-ad60-4ebc-b6a1-c86bc05959fd,001491cf-ad60-4ebc-b6a1-c86bc05959fd,001491cf-ad60-4ebc-b6a1-c86bc05959fd,001491cf-ad60-4ebc-b6a1-c86bc05959fd,001491cf-ad60-4ebc-b6a1-c86bc05959fd,001491cf-ad60-4ebc-b6a1-c86bc05959fd,001491cf-ad60-4ebc-b6a1-c86bc05959fd,...,ffd8f1e1-4a98-456d-a9e6-e06251cdbea9,ffd8f1e1-4a98-456d-a9e6-e06251cdbea9,ffd8f1e1-4a98-456d-a9e6-e06251cdbea9,ffd8f1e1-4a98-456d-a9e6-e06251cdbea9,ffd8f1e1-4a98-456d-a9e6-e06251cdbea9,ffd8f1e1-4a98-456d-a9e6-e06251cdbea9,ffd8f1e1-4a98-456d-a9e6-e06251cdbea9,ffd8f1e1-4a98-456d-a9e6-e06251cdbea9,ffd8f1e1-4a98-456d-a9e6-e06251cdbea9,ffd8f1e1-4a98-456d-a9e6-e06251cdbea9
accountId,account.00ae636f3f29443490716d28017686d2,account.04550355e52549ea99457094928c0316,account.09c7523488a6456abbee4404309fa9b4,account.11232838286e42cab753a80e7d67dbef,account.1ddef90833c34149a8aeef6d6b566e26,account.23931a30c48041c180cccabc6d663610,account.263c3e56b22c4f3d8066119499641066,account.297f06b4123644f7b6b8e711a78a951c,account.363b850923a74180a4eca2205182d331,account.36cf230a19fa4a0080f876af0107bff7,...,account.df6cd7f0c2564f8ba411c493b1cb1887,account.e0818362272f4d8cba82c51e1e8efcd9,account.e32f77c6cde2461eb42e81361c4b1bf1,account.e9b488acacc041d2a8f181afc1f29812,account.e9c13c0ecb9b4aa2a591289b32e15bde,account.ea825a4d327c4f5095718884baf72158,account.f0a6f8d1e24c4b1881bb826ac3d82fd0,account.f0d3c4aaa34142eeb9535ba6e8142e35,account.f54aaf14429943cbab09c30ebc66812b,account.fdbaf316a9cf4cd58e3d57ab14771ac2
drop_time,2026-02-14 17:41:28.028000+00:00,2026-02-14 17:41:09.212000+00:00,2026-02-14 17:41:12.486000+00:00,2026-02-14 17:40:56.542000+00:00,2026-02-14 17:41:12.151000+00:00,2026-02-14 17:41:27.281000+00:00,2026-02-14 17:41:20.423000+00:00,2026-02-14 17:41:35.330000+00:00,2026-02-14 17:40:42.139000+00:00,2026-02-14 17:40:49.557000+00:00,...,2026-02-10 05:40:57.324000+00:00,2026-02-10 05:41:04.771000+00:00,2026-02-10 05:40:57.456000+00:00,2026-02-10 05:40:33.075000+00:00,2026-02-10 05:41:09.312000+00:00,2026-02-10 05:40:50.916000+00:00,2026-02-10 05:41:50.221000+00:00,2026-02-10 05:41:02.152000+00:00,2026-02-10 05:41:09.010000+00:00,2026-02-10 05:40:48.555000+00:00
drop_x,609714.875,106241.0313,196902.3125,414748.3125,611547.1875,601775.6875,701337.75,562870.9375,151178.6406,155545.0313,...,361877.5313,215387.375,302229.125,584996.875,149395.75,411028.6875,162343.9688,345719.375,689577.0,395164.1875
drop_y,595243.125,377565.2188,384114.3125,478791.6875,490069.5625,594725.6875,463816.875,287263.8125,592034.6875,611592.625,...,404218.4063,395918.4375,396509.8125,129427.1172,314229.9063,300776.6875,608055.0,174894.9375,329620.4688,286473.25
drop_z,315.407959,268.444855,840.63385,2930.881836,753.182007,1124.770142,741.512207,1809.783203,1167.249023,697.719788,...,4175.294922,257.935547,653.115051,24740.5332,3580.060303,1020.7995,96.639946,2277.93457,1273.135498,1414.935303
parachute_distance,411.435394,679.113464,613.050049,321.474152,327.853821,400.153625,336.71228,553.56488,307.625397,412.668945,...,368.291626,382.914581,329.795319,315.013245,350.744446,384.351471,758.707886,458.150116,771.994019,348.840973
drop_distance_from_path,152799.087565,23043.979341,23919.139096,52655.253437,47827.076519,152931.603036,14330.748597,150327.179067,187040.307284,206176.433305,...,105047.965282,103215.731265,99977.138618,179314.749643,24516.640241,460.514518,317484.95957,123339.924504,16072.284061,14050.474515
early_enemy_density,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
vehicle_use_ratio,0.185185,0.138462,0.451128,0.184211,0.072464,0.248649,0.3,0.248705,0.364865,0.173333,...,0.163399,0.325,0.486957,0.27972,0.196721,0.044118,0.16129,0.272727,0.415385,0.044118


In [8]:
features_match_user.isnull().sum()

matchId                      0
accountId                    0
drop_time                    0
drop_x                       0
drop_y                       0
drop_z                       0
parachute_distance           0
drop_distance_from_path      0
early_enemy_density          0
vehicle_use_ratio            0
bluezone_exposure_ratio      0
altitude_variance            0
altitude_std                 0
pos_samples                  0
max_vehicle_distance         0
safezone_proximity_mean     77
safezone_edge_ratio         77
rotation_timing_score      543
dtype: int64

In [9]:
features_match_user['safezone_proximity_mean']

0         0.672490
1         0.599718
2         0.410171
3         0.138339
4         0.561798
            ...   
148799    0.326093
148800    0.478814
148801    0.586898
148802    0.523011
148803    0.360424
Name: safezone_proximity_mean, Length: 148804, dtype: float64

In [10]:
#전체 행 갯수, 평균, 표준편차, 최솟값, 사분위수, 최댓값 확인 
features_match_user.describe()

,drop_x,drop_y,drop_z,parachute_distance,drop_distance_from_path,early_enemy_density,vehicle_use_ratio,bluezone_exposure_ratio,altitude_variance,altitude_std,pos_samples,max_vehicle_distance,safezone_proximity_mean,safezone_edge_ratio,rotation_timing_score
count,148804.000000,148804.000000,148804.000000,148804.000000,148804.000000,148804.000000,148804.0,148804.0,1.488040e+05,148804.000000,148804.000000,148804.000000,148727.000000,148727.000000,148261.000000
mean,413629.111658,377864.961863,1622.113837,388.640357,119130.062632,0.078398,0.173397,0.046643,1.410313e+09,35904.051406,95.630541,196917.468286,0.562609,0.119245,0.023034
std,158082.836485,137773.961679,2315.665024,128.289464,100009.953387,0.319285,0.108356,0.090072,8.924404e+08,11009.651904,49.324998,128598.725435,21.134362,0.149203,0.042098
min,7231.764648,2598.840088,-2650.556152,0.000000,0.595576,0.000000,0.0,0.0,0.000000e+00,0.000000,6.000000,278.208074,0.050622,0.000000,0.004658
25%,326445.921875,286278.117225,396.394829,321.483574,41362.440933,0.000000,0.086957,0.0,7.557627e+08,27491.137971,50.000000,100782.772474,0.342228,0.000000,0.007792
50%,414426.625000,349043.656300,1113.026795,348.492996,93728.349222,0.000000,0.151515,0.0,1.102166e+09,33198.892206,96.000000,149639.533407,0.454998,0.051282,0.011050
75%,549260.390625,474461.656300,2102.924683,407.402893,170757.874014,0.000000,0.24,0.052632,1.838897e+09,42882.357609,136.000000,272818.722159,0.568167,0.208333,0.024631
max,775228.187500,748743.750000,34172.160160,7848.220703,575073.668996,6.000000,0.8,0.666667,6.146258e+09,78398.071123,210.000000,748372.612203,6904.902355,1.000000,0.955357


In [11]:
features_match_user.nunique()

matchId                      2223
accountId                   80098
drop_time                  148320
drop_x                     147461
drop_y                     147502
drop_z                     148459
parachute_distance         146687
drop_distance_from_path    148804
early_enemy_density             7
vehicle_use_ratio            6172
bluezone_exposure_ratio      4685
altitude_variance          148798
altitude_std               148798
pos_samples                   204
max_vehicle_distance       148804
safezone_proximity_mean    148727
safezone_edge_ratio          6468
rotation_timing_score        4450
dtype: int64

In [12]:
features_match_user.to_csv("features_match_user.csv")

PermissionError: [Errno 13] Permission denied: 'features_match_user.csv'

In [ ]:
df_posi = pd.read_csv("C:/Users/qkrtl/10th/00_Project/04_final/02_parquet_file/03_mapwise_logs_kakao+steam_20260211_20260219/Erangel/LogPlayerPosition.csv")

C:\Users\qkrtl\AppData\Local\Temp\ipykernel_14656\926736690.py:1: DtypeWarning: Columns (0: character_inSpecialZone) have mixed types. Specify dtype option on import or set low_memory=False.
  df_posi = pd.read_csv("C:/Users/qkrtl/10th/00_Project/04_final/02_parquet_file/03_mapwise_logs_kakao+steam_20260211_20260219/Erangel/LogPlayerPosition.csv")


In [ ]:
df_posi.info()

<class 'pandas.DataFrame'>
RangeIndex: 14235236 entries, 0 to 14235235
Data columns (total 39 columns):
 #   Column                       Dtype  
---  ------                       -----  
 0   matchId                      str    
 1   _D                           str    
 2   _T                           str    
 3   common_isGame                float64
 4   character_name               str    
 5   character_teamId             float64
 6   character_health             float64
 7   character_location_x         float64
 8   character_location_y         float64
 9   character_location_z         float64
 10  character_ranking            float64
 11  character_individualRanking  float64
 12  character_accountId          str    
 13  character_isInBlueZone       bool   
 14  character_isInRedZone        bool   
 15  character_inSpecialZone      str    
 16  character_isInVehicle        bool   
 17  character_zone               str    
 18  character_type               str    
 19  character

In [ ]:
df_posi.describe()

,common_isGame,character_teamId,character_health,character_location_x,character_location_y,character_location_z,character_ranking,character_individualRanking,elapsedTime,numAlivePlayers,vehicle_seatIndex,vehicle_healthPercent,vehicle_feulPercent,vehicle_altitudeAbs,vehicle_altitudeRel,vehicle_velocity,vehicle_location_x,vehicle_location_y,vehicle_location_z,src_date
count,1.423524e+07,1.423524e+07,1.423524e+07,1.423524e+07,1.423524e+07,1.423524e+07,1.423524e+07,1.423524e+07,1.423524e+07,1.423524e+07,2.705261e+06,2.705261e+06,2.705261e+06,2.705261e+06,2.705261e+06,2.705261e+06,2.705261e+06,2.705261e+06,2.705261e+06,1.423524e+07
mean,2.101386e+00,1.769924e+01,8.823241e+01,4.038437e+05,3.807217e+05,1.081970e+04,9.374070e-02,3.227146e+00,5.391468e+02,4.692369e+01,3.849096e-01,7.096114e+01,4.983076e+01,2.964742e+01,2.450175e+01,4.352218e+03,4.091800e+05,3.877234e+05,3.378068e+04,2.026022e+07
std,1.826803e+00,4.050876e+01,2.669138e+01,1.675060e+05,1.480544e+05,3.241627e+04,1.194826e+00,1.253599e+01,4.318415e+02,1.972753e+01,8.123630e-01,4.116701e+01,3.441355e+01,6.601207e+02,5.755133e+02,5.226336e+03,1.742400e+05,1.661626e+05,5.948980e+04,2.618528e+00
min,0.000000e+00,1.000000e+00,0.000000e+00,-1.689991e+05,-1.689981e+05,-2.708835e+05,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,-1.000000e+00,0.000000e+00,0.000000e+00,-1.087415e+03,0.000000e+00,0.000000e+00,-1.689991e+05,-1.689981e+05,-9.173694e+04,2.026021e+07
25%,1.000000e+00,5.000000e+00,9.429676e+01,2.815686e+05,2.786052e+05,5.887318e+02,0.000000e+00,0.000000e+00,1.670000e+02,3.400000e+01,0.000000e+00,4.424051e+01,1.460000e+01,0.000000e+00,0.000000e+00,1.130082e+03,2.857327e+05,2.785412e+05,7.325623e+02,2.026021e+07
50%,1.500000e+00,9.000000e+00,1.000000e+02,3.960988e+05,3.725639e+05,1.465000e+03,0.000000e+00,0.000000e+00,4.570000e+02,4.700000e+01,0.000000e+00,9.755762e+01,5.579825e+01,0.000000e+00,0.000000e+00,2.270122e+03,4.105320e+05,3.814278e+05,2.183327e+03,2.026022e+07
75%,3.000000e+00,1.400000e+01,1.000000e+02,5.429770e+05,4.811572e+05,3.491510e+03,0.000000e+00,0.000000e+00,8.480000e+02,6.000000e+01,0.000000e+00,1.000000e+02,8.010585e+01,0.000000e+00,0.000000e+00,3.440368e+03,5.363930e+05,4.844779e+05,1.007096e+04,2.026022e+07
max,1.050000e+01,2.220000e+02,1.000000e+02,9.849989e+05,9.849991e+05,1.502700e+05,3.200000e+01,1.000000e+02,1.969000e+03,1.000000e+02,4.000000e+00,1.000000e+02,1.000000e+02,2.982714e+04,2.000000e+04,9.383472e+04,9.849989e+05,9.849991e+05,1.500000e+05,2.026022e+07


In [ ]:
df_posi['elapsedTime'].count()

np.int64(14235236)

In [ ]:
df_posi["_D"] = pd.to_datetime(df_posi["_D"])
df_posi = df_posi.sort_values(["matchId", "character_name", "_D"])

df_posi["dt_sec"] = (
    df_posi.groupby(["matchId", "character_name"])["_D"]
      .diff()
      .dt.total_seconds()
)

df_posi["dt_sec"].describe()

count    1.408501e+07
mean     1.020241e+01
std      8.394380e+00
min      7.100000e-02
25%      9.989000e+00
50%      9.998000e+00
75%      1.001200e+01
max      1.006779e+03
Name: dt_sec, dtype: float64

In [ ]:
sample = (
    df_posi[df_posi["matchId"] == df_posi["matchId"].iloc[0]]
    .copy()
)

name = sample["character_name"].dropna().iloc[0]
sample = sample[sample["character_name"] == name].copy()

sample["_D"] = pd.to_datetime(sample["_D"], errors="coerce")
sample = sample.sort_values("_D")

cols = ["_D", "elapsedTime", "character_location_x", "character_location_y", "character_location_z"]
print(sample[cols].head(20))

                                      _D  elapsedTime  character_location_x  \
3667275 2026-02-14 17:38:44.835000+00:00          0.0         117630.750000   
3667337 2026-02-14 17:38:54.840000+00:00          0.0         117630.796875   
3667398 2026-02-14 17:39:04.862000+00:00          0.0         117630.796875   
3667460 2026-02-14 17:39:14.849000+00:00          0.0         117333.593750   
3667522 2026-02-14 17:39:24.837000+00:00          0.0         115478.531250   
3667584 2026-02-14 17:39:34.858000+00:00          0.0         115733.703125   
3667646 2026-02-14 17:39:44.845000+00:00          1.0        -140359.328125   
3667708 2026-02-14 17:39:54.852000+00:00         11.0           1724.531250   
3667770 2026-02-14 17:40:04.854000+00:00         21.0         143730.656250   
3667832 2026-02-14 17:40:14.835000+00:00         31.0         285408.125000   
3667894 2026-02-14 17:40:24.857000+00:00         41.0         427687.937500   
3667956 2026-02-14 17:40:34.865000+00:00         51.

In [ ]:
df_posi['elapsedTime']



3667275       0.0
3667337       0.0
3667398       0.0
3667460       0.0
3667522       0.0
            ...  
1535377    1424.0
1535385    1434.0
1535393    1444.0
1535400    1454.0
1535406    1464.0
Name: elapsedTime, Length: 14235236, dtype: float64


LogPlayerPosition은 10초마다 로그를 찍는다.

In [ ]:
features_match_user['drop_distance_from_path']

0         152799.087565
1          23043.979341
2          23919.139096
3          52655.253437
4          47827.076519
              ...      
148799       460.514518
148800    317484.959570
148801    123339.924504
148802     16072.284061
148803     14050.474515
Name: drop_distance_from_path, Length: 148804, dtype: float64

In [19]:
## clustering
# by user style

# 1. match - user 스타일 벡터 정리
# 2. user profile 생성
# 3. 클러스터링


features_match_user.isnull().count()

matchId                    148804
accountId                  148804
drop_time                  148804
drop_x                     148804
drop_y                     148804
drop_z                     148804
parachute_distance         148804
drop_distance_from_path    148804
early_enemy_density        148804
vehicle_use_ratio          148804
bluezone_exposure_ratio    148804
altitude_variance          148804
altitude_std               148804
pos_samples                148804
max_vehicle_distance       148804
safezone_proximity_mean    148804
safezone_edge_ratio        148804
rotation_timing_score      148804
dtype: int64

In [14]:
features_match_user.info()

<class 'pandas.DataFrame'>
RangeIndex: 148804 entries, 0 to 148803
Data columns (total 18 columns):
 #   Column                   Non-Null Count   Dtype              
---  ------                   --------------   -----              
 0   matchId                  148804 non-null  str                
 1   accountId                148804 non-null  str                
 2   drop_time                148804 non-null  datetime64[us, UTC]
 3   drop_x                   148804 non-null  float64            
 4   drop_y                   148804 non-null  float64            
 5   drop_z                   148804 non-null  float64            
 6   parachute_distance       148804 non-null  float64            
 7   drop_distance_from_path  148804 non-null  float64            
 8   early_enemy_density      148804 non-null  int64              
 9   vehicle_use_ratio        148804 non-null  Float64            
 10  bluezone_exposure_ratio  148804 non-null  Float64            
 11  altitude_variance       

In [ ]:
## 수집시작 : 2월 11일 ~ 2월 18일
# 수집한 텔레메트리 안에는 2월 6일부터의 데이터가 쌓여있음
features_match_user.groupby(features_match_user['drop_time'].dt.date).size()


drop_time
2026-02-06     2028
2026-02-07     3273
2026-02-08     5802
2026-02-09     6610
2026-02-10     8279
2026-02-11    13120
2026-02-12    16425
2026-02-13    16604
2026-02-14    15836
2026-02-15    19074
2026-02-16    15003
2026-02-17    11784
2026-02-18    14966
dtype: int64